# ASO Acute Toxicity Prediction Pipeline
### Quiver Bioscience — Sub-workstream 3

**How to use:**
1. Place `ASO_tox_prediction_pipeline.ipynb` and `aso_tox_gbr_model.pkl` in the **same folder**
2. Set `INPUT_CSV` in Step 1 to your sequences file
3. Click **Run All** — done

```
your_folder/
├── ASO_tox_prediction_pipeline.ipynb   ← this notebook
├── aso_tox_gbr_model.pkl               ← pre-trained model (auto-detected)
└── your_sequences.csv                  ← your input
```

**Output:** `tox_predictions/target_tox_predictions.csv` with columns:
`Sequence`, `Hagedorn_predict_toxscore`, `GBR_predict_toxscore` (+ all extracted features)

**Model loading logic:**
- If `aso_tox_gbr_model.pkl` found in same folder → loads it, training is skipped
- If not found → trains from scratch using Ionis patent datasets, saves pkl for next time


## Step 0 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from pathlib import Path
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from IPython.display import display

print("✓ All imports successful.")


## Step 1 — Configuration
These are the **only lines you need to change** between runs.


In [ ]:
# ── INPUT ─────────────────────────────────────────────────────────────────────
INPUT_CSV = "input_sequences.csv"    # ← change to your file

# Column name containing sequences. Set to None to auto-detect.
# Auto-detects if column is named: 'Sequence', 'sequence', or 'ASO'
SEQUENCE_COLUMN = None

# ── MODEL: auto-detected from same folder as this notebook ────────────────────
NOTEBOOK_DIR = Path().resolve()
MODEL_PATH   = NOTEBOOK_DIR / "aso_tox_gbr_model.pkl"

# ── TRAINING DATA: only needed if pkl is missing ──────────────────────────────
TRAIN_CSV_1 = NOTEBOOK_DIR / "UBE3A_ATS_ASO_info_v2.csv"
TRAIN_CSV_2 = NOTEBOOK_DIR / "GFAP_ASO_withICV_score_v2.csv"

# ── OUTPUT: saved to tox_predictions/target_tox_predictions.csv ───────────────
OUTPUT_DIR  = NOTEBOOK_DIR / "tox_predictions"
OUTPUT_CSV  = OUTPUT_DIR / "target_tox_predictions.csv"

# ── TOX THRESHOLD ─────────────────────────────────────────────────────────────
TOX_THRESHOLD = 1.0   # Ionis scale: >1.0 = Toxic  (Roche scale: 4.0)

# ─────────────────────────────────────────────────────────────────────────────
print(f"Notebook folder : {NOTEBOOK_DIR}")
print(f"Model path      : {MODEL_PATH.name}  ({'EXISTS — training will be skipped' if MODEL_PATH.exists() else 'NOT FOUND — will train and save'})")
print(f"Input CSV       : {INPUT_CSV}")
print(f"Output CSV      : {OUTPUT_CSV}")
print(f"Tox threshold   : >{TOX_THRESHOLD} = Toxic")


## Step 2 — Load Model (or Train if pkl Not Found)
No action needed. This cell handles everything automatically.


In [ ]:
FEATURE_COLS = [
    'MaxLength_A', 'MaxLength_T', 'MaxLength_G', 'MaxLength_C',
    'Number_A',    'Number_C',    'Number_T',    'Number_G',
    'Length_Gfree_5', 'Length_Gfree_3'
]

if MODEL_PATH.exists():
    # ── PKL FOUND: load and skip training ────────────────────────────────────
    model = joblib.load(MODEL_PATH)
    print(f"✓ Model loaded from:  {MODEL_PATH.name}")
    print(f"  Training skipped.")
    p = model.get_params()
    print(f"  Params: n_estimators={p['n_estimators']}, max_depth={p['max_depth']}, "
          f"learning_rate={p['learning_rate']}, random_state={p['random_state']}")

else:
    # ── PKL NOT FOUND: train from scratch then save ───────────────────────────
    print("pkl not found — training from scratch...")
    df1 = pd.read_csv(TRAIN_CSV_1)
    df2 = pd.read_csv(TRAIN_CSV_2)
    df  = pd.concat([df1, df2])

    df_shuffle = df.sample(frac=1, random_state=42)
    X_train = df_shuffle[FEATURE_COLS]
    y_train = df_shuffle['Acute_tolerance_score_mice']
    print(f"  Training set: {len(df_shuffle)} ASOs")

    base_model = GradientBoostingRegressor(random_state=0)
    param_grid = {
        'n_estimators':  range(80, 120, 10),
        'max_depth':     [3],
        'learning_rate': [0.05, 0.1, 0.15, 0.2]
    }
    grid = GridSearchCV(estimator=base_model, param_grid=param_grid)
    grid.fit(X_train, y_train)
    model = grid.best_estimator_

    print(f"  Best params : {grid.best_params_}")
    print(f"  CV score    : {grid.best_score_:.4f}")

    joblib.dump(model, MODEL_PATH)
    print(f"\n✓ Model trained and saved to: {MODEL_PATH.name}")
    print(f"  Next run will load this pkl — no retraining needed.")


## Step 3 — Feature Extraction & Hagedorn Score

Two scores are computed per sequence:

| Score | Formula | Source |
|---|---|---|
| `Hagedorn_predict_toxscore` | `136.0430 − 3.1263·A − 5.1100·C − 4.7217·T − 10.1264·G + 1.3577·Gfree3` | Defined as `baseModel()` in original notebook |
| `GBR_predict_toxscore` | GradientBoostingRegressor trained on Ionis UBE3A + GFAP datasets | Zhang GBR model |

> **Note for downstream code:** The T coefficient is `4.7217` in the Python notebook (`baseModel()`).
> The R version uses `4.7247`. Both are reproduced here using the Python notebook value.


In [ ]:
def extract_features(seq):
    """Extract 10 GBR model features. Reproduces ASO_feature_extraction_ML.m exactly."""
    seq = seq.upper().strip()
    n   = len(seq)
    f   = {}

    # Max consecutive run length per nucleotide (capped at 6)
    for base in ['A', 'T', 'G', 'C']:
        max_run = 0
        for r in range(1, 7):
            if base * r in seq:
                max_run = r
        f[f'MaxLength_{base}'] = max_run

    # Total nucleotide counts
    for b in ['A', 'T', 'G', 'C']:
        f[f'Number_{b}'] = seq.count(b)

    # Gfree5: nt before first G from 5' end
    Gfree5 = n
    for k in range(n):
        if seq[k] == 'G':
            Gfree5 = k
            break
    f['Length_Gfree_5'] = Gfree5

    # Gfree3: nt after last G from 3' end
    Gfree3 = n
    for k in range(n - 1, -1, -1):
        if seq[k] == 'G':
            Gfree3 = n - 1 - k
            break
    f['Length_Gfree_3'] = Gfree3

    return f


def hagedorn_score(f):
    """
    Hagedorn (baseModel) formula — exactly as defined in ASO_tox_gradient_boosting_vf.ipynb:
        136.0430 - 3.1263*Number_A - 5.1100*Number_C - 4.7217*Number_T
                 - 10.1264*Number_G + 1.3577*Length_Gfree_3
    """
    return (136.0430
            - 3.1263  * f['Number_A']
            - 5.1100  * f['Number_C']
            - 4.7217  * f['Number_T']
            - 10.1264 * f['Number_G']
            + 1.3577  * f['Length_Gfree_3'])


print("✓ Feature extraction and Hagedorn score functions defined.")


## Step 4 — Load & Validate Input Sequences

In [ ]:
df_input = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df_input)} rows from {INPUT_CSV}")
display(df_input.head(5))

# Auto-detect sequence column
if SEQUENCE_COLUMN is not None:
    seq_col = SEQUENCE_COLUMN
elif 'Sequence' in df_input.columns:
    seq_col = 'Sequence'
elif 'sequence' in df_input.columns:
    seq_col = 'sequence'
elif 'ASO' in df_input.columns:
    seq_col = 'ASO'
else:
    raise ValueError(
        f"Cannot find sequence column. Columns: {list(df_input.columns)}. "
        f"Set SEQUENCE_COLUMN manually in Step 1."
    )

sequences = df_input[seq_col].astype(str).str.strip().str.upper()
print(f"\nSequence column : '{seq_col}'")

invalid = [(i, s, set(s) - set('ATGC')) for i, s in enumerate(sequences) if set(s) - set('ATGC')]
if invalid:
    print(f"⚠️  {len(invalid)} sequences contain unexpected characters:")
    for idx, seq, bad in invalid[:5]:
        print(f"   Row {idx}: '{seq}'  unexpected: {bad}")
else:
    L = sequences.str.len()
    print(f"✓ All {len(sequences)} sequences valid  (length {L.min()}–{L.max()} nt, mean {L.mean():.1f})")


## Step 5 — Extract Features & Compute Both Scores

In [ ]:
feature_records  = [extract_features(seq) for seq in sequences]
df_features      = pd.DataFrame(feature_records)[FEATURE_COLS]

hagedorn_scores  = [hagedorn_score(f) for f in feature_records]
gbr_scores       = model.predict(df_features.values)
tox_labels       = ['Toxic' if s > TOX_THRESHOLD else 'Non-toxic' for s in gbr_scores]

print(f"✓ Both scores computed for {len(sequences)} sequences.")
print(f"\n  Hagedorn score range : {min(hagedorn_scores):.3f} – {max(hagedorn_scores):.3f}")
print(f"  GBR score range      : {gbr_scores.min():.3f} – {gbr_scores.max():.3f}")
print(f"  Toxic (GBR >{TOX_THRESHOLD})  : {sum(s > TOX_THRESHOLD for s in gbr_scores)} / {len(gbr_scores)}")


## Step 6 — Assemble Results

In [ ]:
df_out = df_input.copy()

# ── Three required columns (exact names per colleague's spec) ─────────────────
df_out['Sequence']                = sequences.values
df_out['Hagedorn_predict_toxscore'] = np.round(hagedorn_scores, 4)
df_out['GBR_predict_toxscore']      = np.round(gbr_scores, 9)

# ── Additional useful columns ─────────────────────────────────────────────────
df_out['Tox_label'] = tox_labels
for col in FEATURE_COLS:
    df_out[col] = df_features[col].values

# Sort by GBR score ascending (least toxic first)
df_out = df_out.sort_values('GBR_predict_toxscore', ascending=True).reset_index(drop=True)
df_out.index = df_out.index + 1
df_out.index.name = 'Rank'

# Preview
preview_cols = ['Sequence', 'Hagedorn_predict_toxscore', 'GBR_predict_toxscore', 'Tox_label']
display(df_out[preview_cols].style
        .background_gradient(subset=['GBR_predict_toxscore'], cmap='RdYlGn_r')
        .set_caption(f"Ranked ASO Candidates  |  GBR threshold >{TOX_THRESHOLD} = Toxic"))


## Step 7 — Export to `tox_predictions/target_tox_predictions.csv`

In [ ]:
# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_out.to_csv(OUTPUT_CSV)

print(f"✓ Saved: {OUTPUT_CSV}")
print(f"  {len(df_out)} sequences × {len(df_out.columns)} columns")
print(f"\nRequired columns confirmed:")
for col in ['Sequence', 'Hagedorn_predict_toxscore', 'GBR_predict_toxscore']:
    print(f"  ✓  {col}")


## Optional — Feature Importance Plot

In [ ]:
importances = model.feature_importances_
idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#9C02FA' if i == idx[-1] else '#CAA1FF' for i in idx]
ax.barh([FEATURE_COLS[i] for i in idx], importances[idx], color=colors)
ax.set_xlabel("Feature Importance")
ax.set_title("GBR Model — Feature Importances", fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(NOTEBOOK_DIR / "GBR_feature_importance.png", dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: GBR_feature_importance.png")


---
## Appendix — Generate Sample Input CSV
Run this cell to create a test file and verify the full pipeline end-to-end.


In [ ]:
sample = pd.DataFrame({'Sequence': [
    "GCACTTGAATTTCACGTTGT",
    "GGTGAATCTTTATTAAAC",
    "GGTGAATCTTTATTAAACTA",
    "TTGCTCCACCTTGGCCTGGCA",
    "GCGCAGGTAATCCCAAAAGCG",
    "AGCGCAGGTAATCCCAAAAGC",
]})
sample_path = NOTEBOOK_DIR / "input_sequences.csv"
sample.to_csv(sample_path, index=False)
print(f"✓ Created: {sample_path.name}  ({len(sample)} sequences)")
print("  Set INPUT_CSV = 'input_sequences.csv' in Step 1 and Run All.")
display(sample)
